# Induction Evals
We evaluate that the induction case is strong enough to drive a wedge between LLM performance in this notebook.

In [2]:
"""The following generates the Quiz all our models will be evaluated on."""
import itertools
import string
import numpy as np

from dotenv import load_dotenv

load_dotenv("keys.env", verbose=True)

from smolbench.induction.chromatic import (
	ChromaticIntervalsConfig,
	Prompter,
	get_random_exclusive_quiz,
	anneal_intervals
)
from smolbench.evals import (
	Marks
)
from smolbench.evals.openrouter import (
	evaluate
)

template = string.Template(
	"You are a Boolean classifier.\n"
	"\n"
	"Task: determine whether the statement in the Question is logically "
	"possible given the Context.\n"
	"\n"
	"Output format:\n"
	"Return exactly one of these two strings and nothing else:\n"
	"True\n"
	"False\n"
	"\n"
	"Do not output any explanation, punctuation, quotes, labels, code fences, "
	"or extra whitespace."
	"Stop immediately after writing True or False."
	"\n"
	"Context:\n"
	"There is a ceremonial role called the $role, whose job it is to"
	" head the $parade parade. No one else besides the $role is able to head"
	" the $parade parade. The following lists the people who were $role and"
	" the years they were $role:\n"
	"$positive_info\n"
	"\n"
	"Question:\n"
	"Between the years $start and $end, exclusive of the end, could $color"
	" have headed the $parade parade every year?"
)

def query_gen(
	labels_to_intervals: Dict[Color, Intervals],
	interval_to_label: Dict[Intervals, Color],
	seed: int,
) -> Dict[str, str]:
	"""Generates a series of queries"""
	rng: np.random.Generator = np.random.default_rng(seed)
	# Finds max interval.
	n: int = max(interval[1] for interval in interval_to_label)
	for color, intervals in labels_to_intervals.items():
		# Generates a series of true items.
		for start, end in intervals:
			start, end = np.sort(
				rng.choice(range(start, end + 1), size=2, replace=False)
			)
			yield {"color": color, "start": start, "end": end}, True
		# Generates a false proposition.
		invalid_range: Intervals = anneal_intervals(
			itertools.chain(
				(set(interval_to_label.keys()) - set(itertools.chain(*intervals)))
			)
		)
		for start, end in invalid_range:
			start = rng.choice(range(start, end))
			# Binom with p = intervals / n capped at end for a similar-ish
			# distr. to positive accounts.
			end = min(
				end,
				start
				+ rng.binomial(
					end - start + 1,
					np.mean([len(interval) for interval in interval_to_label]) / n,
				)
				+ 1,
			)
			yield {"color": color, "start": start, "end": end}, False


SEED: int = 1776
intens_quiz, extens_quiz = get_random_exclusive_quiz(
	ChromaticIntervalsConfig(
            n=250,
            intervals=250 // 4,
            colors=45,
            seed=SEED,
        ),
	Prompter(
		template,
		{
			"role": "Twislax",
			"parade": "Gildane",
		},
		query_gen,
	),
)

## Prompt Validation

In [3]:
print(intens_quiz[0].prompt)

You are a Boolean classifier.

Task: determine whether the statement in the Question is logically possible given the Context.

Output format:
Return exactly one of these two strings and nothing else:
True
False

Do not output any explanation, punctuation, quotes, labels, code fences, or extra whitespace.Stop immediately after writing True or False.
Context:
There is a ceremonial role called the Twislax, whose job it is to head the Gildane parade. No one else besides the Twislax is able to head the Gildane parade. The following lists the people who were Twislax and the years they were Twislax:
Vp was Twislax on 24 to 34.
Cj was Twislax on 108 to 120 and 137 to 139.
Gt was Twislax on 35 to 36 and 134 to 135.
Wk was Twislax on 63 to 63.
kn was Twislax on 186 to 190.
tP was Twislax on 232 to 233.
vS was Twislax on 88 to 90.
sU was Twislax on 38 to 41 and 201 to 204.
Ph was Twislax on 0 to 10, 180 to 181, and 225 to 226.
FX was Twislax on 61 to 62, 149 to 160, and 191 to 200.
jp was Twislax

In [4]:
print(extens_quiz[0].prompt)

You are a Boolean classifier.

Task: determine whether the statement in the Question is logically possible given the Context.

Output format:
Return exactly one of these two strings and nothing else:
True
False

Do not output any explanation, punctuation, quotes, labels, code fences, or extra whitespace.Stop immediately after writing True or False.
Context:
There is a ceremonial role called the Twislax, whose job it is to head the Gildane parade. No one else besides the Twislax is able to head the Gildane parade. The following lists the people who were Twislax and the years they were Twislax:
Vp was Twislax on 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, and 34.
Cj was Twislax on 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 137, 138, and 139.
Gt was Twislax on 35, 36, 134, and 135.
Wk was Twislax on 63.
kn was Twislax on 186, 187, 188, 189, and 190.
tP was Twislax on 232 and 233.
vS was Twislax on 88, 89, and 90.
sU was Twislax on 38, 39, 40, 41, 201, 202, 203, and 204.


# Decoder-Only Model
This section tests classical decoder-only models.

In [5]:
decode_intens_eval: Marks = evaluate(intens_quiz, "mistralai/mistral-7b-instruct-v0.1", SEED)

0 0 0
0 1 0
' False. Vp was Twislax on years 24 to 34, which is not inclusive of or overlaps with the years 163 to 166.' is not a bool.
0 1 1
0 2 1
0 3 1
1 3 1
1 4 1
2 4 1
3 4 1
3 5 1
4 5 1
4 6 1
5 6 1
5 7 1
' False.

tP was Twislax only in the years 232 and 233.' is not a bool.
5 7 2
5 8 2
5 9 2
6 9 2
6 10 2
6 11 2
7 11 2
7 12 2
7 13 2
7 14 2
8 14 2
8 15 2
8 16 2
8 17 2
9 17 2
10 17 2
10 18 2
11 18 2
11 19 2
11 20 2
12 20 2
' False.

The context states that dr was not Twislax during the years 166 to 169.' is not a bool.
12 20 3
' False.
Im is not listed as Twislax during the years 89 and 90.' is not a bool.
12 20 4
12 21 4
13 21 4
13 22 4
14 22 4
15 22 4
15 23 4
15 24 4
15 25 4
15 26 4
16 26 4
16 27 4
17 27 4
17 28 4
' False.
ld was Twislax only in the year 121.' is not a bool.
17 28 5
18 28 5
18 29 5
19 29 5
19 30 5
19 31 5
20 31 5
20 32 5
21 32 5
21 33 5
22 33 5
' False.
The context states that xe was not Twislax during the years 40 to 42.' is not a bool.
22 33 6
22 34 6
23 34 6
' F

In [6]:
decode_extens_eval: Marks = evaluate(extens_quiz, "mistralai/mistral-7b-instruct-v0.1", SEED)

0 0 0
1 0 0
2 0 0
2 1 0
2 2 0
2 3 0
2 4 0
3 4 0
4 4 0
4 5 0
5 5 0
6 5 0
7 5 0
7 6 0
8 6 0
8 7 0
8 8 0
9 8 0
10 8 0
' True

(Note: According to the context, sU was Twislax in the years 38, 39, 40, 41, 201, 202, 203, and 204. The question asks about the years between 203 and 205, which includes the years 203 and 204 but excludes the year 205. Since sU was Twislax in the year 203 and 204, the answer is "True".)' is not a bool.
10 8 1
11 8 1
11 9 1
11 10 1
11 11 1
12 11 1
' True

(FX was Twislax in years 61, 62, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 191, 192, 193, 194, 195, 196, 197, 198, 199, and 200. The statement asks for every year between 61 and 62, so it is asking for years 61, 151, 152, and 153. All of these years are listed in the context as years that FX was Twislax.)' is not a bool.
12 11 2
13 11 2
13 12 2
14 12 2
15 12 2
15 13 2
16 13 2
17 13 2
17 14 2
18 14 2
19 14 2
' False.
Im is not listed as Twislax during the years between 89 and 90.' is not a bool.
1

In [7]:
print(decode_intens_eval.correct, decode_intens_eval.incorrect, decode_intens_eval.invalid)
print(decode_extens_eval.correct, decode_extens_eval.incorrect, decode_extens_eval.invalid)

39 56 12
60 38 9


## CoT Model
This section tests a CoT model.

In [8]:
cot_intens_eval: Marks = evaluate(intens_quiz, "deepseek/deepseek-r1-distill-qwen-14b", SEED)

0 0 0
{"error":{"message":"No endpoints found for deepseek/deepseek-r1-distill-qwen-14b.","code":404},"user_id":"user_2rvYe1mPleD8Rt0MF9krjXl3XuQ"}


HTTPError: 404 Client Error: Not Found for url: https://openrouter.ai/api/v1/chat/completions

In [ ]:
cot_extens_eval: Marks = evaluate(extens_quiz, "deepseek/deepseek-r1-distill-qwen-14b", SEED)